## 0. Install Required Dependencies

**Run this cell first if you encounter import errors**

In [1]:
import sys
import subprocess

# Install required dependencies
dependencies = ['accelerate>=0.26.0', 'datasets']

for dep in dependencies:
    print(f"Installing {dep}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])
    
print("✅ All dependencies installed! Please restart the kernel (Kernel > Restart Kernel) and then run all cells.")

Installing accelerate>=0.26.0...
Installing datasets...
Installing datasets...
✅ All dependencies installed! Please restart the kernel (Kernel > Restart Kernel) and then run all cells.
✅ All dependencies installed! Please restart the kernel (Kernel > Restart Kernel) and then run all cells.


# Fine-tune SentenceTransformer Models for ITSM Tickets
This notebook fine-tunes the **all-mpnet-base-v2** embedding model (and can be adapted for others) using contrastive learning with pseudo-labeled training data from your ITSM tickets.
## Approach
- **Positive pairs**: Tickets from the same category (assumed similar)
- **Negative pairs**: Tickets from different categories (assumed dissimilar)
- **Loss function**: Cosine Similarity Loss (contrastive learning)
- **Base model**: sentence-transformers/all-mpnet-base-v2 (768-dim embeddings)

## 1. Setup and Imports

In [2]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), '..'))

import json
import torch
from datetime import datetime
import logging
from datasets import DatasetDict  # <-- Added here

# Import sentence-transformers
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

/opt/anaconda3/envs/itsm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful
PyTorch version: 2.9.1
Device: CPU


## 2. Configuration

In [3]:
# Training configuration
CONFIG = {
    'base_model': 'sentence-transformers/all-mpnet-base-v2',
    'source_data': '../data/servicenow_incidents_full.json',  # Source incidents
    'output_dir': 'models/all-mpnet-finetuned',
    'epochs': 10,  # Start with fewer epochs, can increase if needed
    'batch_size': 32,
    'learning_rate': 2e-5,
    'warmup_steps': 100,
    'eval_split': 0.1  # 10% for evaluation
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration:
  base_model: sentence-transformers/all-mpnet-base-v2
  training_file: data/training_pairs.json
  output_dir: models/all-mpnet-finetuned
  epochs: 50
  batch_size: 32
  learning_rate: 2e-05
  warmup_steps: 100
  eval_split: 0.1


## 3. Load Training Data

In [ ]:
# Generate training pairs from ServiceNow incidents
import random
from collections import defaultdict

# Load ServiceNow incidents
incidents_file = os.path.join(os.path.dirname(os.getcwd()), 'data', 'servicenow_incidents_full.json')
print(f"Loading incidents from: {incidents_file}")

with open(incidents_file, 'r') as f:
    incidents = json.load(f)

print(f"Loaded {len(incidents)} incidents")

# Group incidents by category
category_groups = defaultdict(list)
for incident in incidents:
    category = incident.get('category', 'Unknown')
    if category and category != '':
        # Create text representation combining short_description and description
        text = f"{incident.get('short_description', '')}. {incident.get('description', '')}"
        category_groups[category].append({
            'id': incident.get('incident_number', incident.get('sys_id', '')),
            'text': text.strip(),
            'category': category
        })

print(f"\nCategories found: {len(category_groups)}")
for cat, items in category_groups.items():
    print(f"  {cat}: {len(items)} incidents")

# Generate positive pairs (same category)
positive_pairs = []
for category, items in category_groups.items():
    if len(items) >= 2:
        # Create pairs within the same category
        for i in range(len(items)):
            for j in range(i + 1, min(i + 6, len(items))):  # Limit pairs per incident
                positive_pairs.append({
                    'ticket1_id': items[i]['id'],
                    'ticket2_id': items[j]['id'],
                    'text1': items[i]['text'],
                    'text2': items[j]['text'],
                    'category1': category,
                    'category2': category
                })

# Generate negative pairs (different categories)
negative_pairs = []
categories = list(category_groups.keys())
for i in range(len(categories)):
    for j in range(i + 1, len(categories)):
        cat1_items = category_groups[categories[i]]
        cat2_items = category_groups[categories[j]]
        
        # Sample random pairs between different categories
        num_pairs = min(len(cat1_items) * 2, len(cat2_items) * 2, 50)
        for _ in range(num_pairs):
            item1 = random.choice(cat1_items)
            item2 = random.choice(cat2_items)
            negative_pairs.append({
                'ticket1_id': item1['id'],
                'ticket2_id': item2['id'],
                'text1': item1['text'],
                'text2': item2['text'],
                'category1': item1['category'],
                'category2': item2['category']
            })

print(f"\n📊 Generated Training Pairs:")
print(f"  Positive pairs: {len(positive_pairs)}")
print(f"  Negative pairs: {len(negative_pairs)}")
print(f"  Total pairs: {len(positive_pairs) + len(negative_pairs)}")

# Save to training_pairs.json for future use
training_data = {
    'positive_pairs': positive_pairs,
    'negative_pairs': negative_pairs,
    'metadata': {
        'num_incidents': len(incidents),
        'num_categories': len(category_groups),
        'generated_on': datetime.now().isoformat()
    }
}

training_pairs_path = os.path.join(os.getcwd(), 'data', 'training_pairs.json')
os.makedirs(os.path.dirname(training_pairs_path), exist_ok=True)
with open(training_pairs_path, 'w') as f:
    json.dump(training_data, f, indent=2)

print(f"\n✅ Training pairs saved to: {training_pairs_path}")

In [4]:
# The training pairs are already loaded from the previous cell
# Just display a summary
print(f"\n📊 Training Data Summary:")
print(f"  Positive pairs: {len(positive_pairs)}")
print(f"  Negative pairs: {len(negative_pairs)}")
print(f"  Total pairs: {len(positive_pairs) + len(negative_pairs)}")

# Show example pairs
if positive_pairs:
    print(f"\n📝 Example Positive Pair (same category):")
    example = positive_pairs[0]
    print(f"  Category: {example['category1']}")
    print(f"  Ticket 1 ({example['ticket1_id']}): {example['text1'][:100]}...")
    print(f"  Ticket 2 ({example['ticket2_id']}): {example['text2'][:100]}...")

if negative_pairs:
    print(f"\n📝 Example Negative Pair (different categories):")
    example = negative_pairs[0]
    print(f"  Category 1: {example['category1']}")
    print(f"  Category 2: {example['category2']}")
    print(f"  Ticket 1 ({example['ticket1_id']}): {example['text1'][:100]}...")
    print(f"  Ticket 2 ({example['ticket2_id']}): {example['text2'][:100]}...")

Loading training data from: /Users/don/DocumentsMac/Codes/itsm-insight-nexus/backend-python/scripts/finetuning/data/training_pairs.json

📊 Training Data:
  Positive pairs: 161
  Negative pairs: 161
  Total pairs: 322

📝 Example Positive Pair:
  Category: Database
  Text 1: Title: Need access to sales DB for the West Description: I have to analyze all US data in the SFA sy...
  Text 2: Title: Need Oracle 10GR2 installed Description: Currently running 10GR1 and need to upgrade to 10GR2...


## 4. Create Training Examples

In [5]:
# Convert to InputExample objects
train_examples = []

# Add positive pairs (label=1.0 for similar)
for pair in positive_pairs:
    train_examples.append(InputExample(
        texts=[pair['text1'], pair['text2']],
        label=1.0
    ))

# Add negative pairs (label=0.0 for dissimilar)
for pair in negative_pairs:
    train_examples.append(InputExample(
        texts=[pair['text1'], pair['text2']],
        label=0.0
    ))

print(f"Created {len(train_examples)} training examples")

# Split into train/eval
import random
random.shuffle(train_examples)
split_idx = int(len(train_examples) * (1 - CONFIG['eval_split']))
eval_examples = train_examples[split_idx:]
train_examples = train_examples[:split_idx]

print(f"\n📊 Data Split:")
print(f"  Training: {len(train_examples)} examples")
print(f"  Evaluation: {len(eval_examples)} examples")

Created 322 training examples

📊 Data Split:
  Training: 289 examples
  Evaluation: 33 examples


## 5. Load Base Model

In [6]:
print(f"Loading base model: {CONFIG['base_model']}")
print("This may take a minute...\n")

model = SentenceTransformer(CONFIG['base_model'])

print("✅ Model loaded successfully")
print(f"\nModel details:")
print(f"  Max sequence length: {model.max_seq_length}")
print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")

2025-11-13 02:44:38,424 - INFO - Use pytorch device_name: mps
2025-11-13 02:44:38,425 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2
2025-11-13 02:44:38,425 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


Loading base model: sentence-transformers/all-mpnet-base-v2
This may take a minute...

✅ Model loaded successfully

Model details:
  Max sequence length: 384
  Embedding dimension: 768
✅ Model loaded successfully

Model details:
  Max sequence length: 384
  Embedding dimension: 768


## 6. Setup Training Components

In [7]:
# Create DataLoader
train_dataloader = DataLoader(
    train_examples, 
    shuffle=True, 
    batch_size=CONFIG['batch_size']
)

# Define loss function (Cosine Similarity Loss for contrastive learning)
train_loss = losses.CosineSimilarityLoss(model)

# Create evaluator
eval_sentences1 = [ex.texts[0] for ex in eval_examples]
eval_sentences2 = [ex.texts[1] for ex in eval_examples]
eval_scores = [ex.label for ex in eval_examples]

evaluator = EmbeddingSimilarityEvaluator(
    eval_sentences1, 
    eval_sentences2, 
    eval_scores,
    name='itsm-eval'
)

# Output directory
output_path = os.path.join(os.getcwd(), CONFIG['output_dir'])
os.makedirs(output_path, exist_ok=True)

print("✅ Training components ready")
print(f"\nTotal training batches: {len(train_dataloader)}")
print(f"Evaluation samples: {len(eval_examples)}")
print(f"Output path: {output_path}")

✅ Training components ready

Total training batches: 10
Evaluation samples: 33
Output path: /Users/don/DocumentsMac/Codes/itsm-insight-nexus/backend-python/scripts/finetuning/models/all-mpnet-finetuned


## 7. Train the Model

⚠️ **Note**: Training on CPU will take 5-15 minutes per epoch. GPU is recommended for faster training.

In [ ]:
print("🚀 Starting training...")
print("=" * 60)
print(f"Epochs: {CONFIG['epochs']}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"Learning rate: {CONFIG['learning_rate']}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print("=" * 60)
print()

# Train
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=CONFIG['epochs'],
    evaluator=evaluator,
    evaluation_steps=len(train_dataloader) // 2,  # Evaluate twice per epoch
    warmup_steps=CONFIG['warmup_steps'],
    output_path=output_path,
    optimizer_params={'lr': CONFIG['learning_rate']},
    save_best_model=True,
    show_progress_bar=True
)

print("\n" + "=" * 60)
print("✅ Training complete!")
print("=" * 60)

🚀 Starting training...
Epochs: 50
Batch size: 32
Learning rate: 2e-05
Device: CPU

2025-11-13 02:57:08,745 - INFO - EmbeddingSimilarityEvaluator: Evaluating the model on the itsm-eval dataset in epoch 28.5 after 285 steps:
2025-11-13 02:57:09,604 - INFO - Cosine-Similarity:	Pearson: 0.9993	Spearman: 0.8467


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/anaconda3/envs/itsm/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/itsm/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Itsm-eval Pearson Cosine,Itsm-eval Spearman Cosine
5,No log,No log,0.009841,-0.019539
10,No log,No log,0.016625,-0.013026
15,No log,No log,0.005384,-0.045592
20,No log,No log,-0.023137,-0.065131
25,No log,No log,-0.021509,-0.026053
30,No log,No log,0.026526,0.032566
35,No log,No log,0.105810,0.149802
40,No log,No log,0.184055,0.201907
45,No log,No log,0.248311,0.267038
50,No log,No log,0.302709,0.358222


2025-11-13 02:44:49,250 - INFO - EmbeddingSimilarityEvaluator: Evaluating the model on the itsm-eval dataset in epoch 0.5 after 5 steps:
2025-11-13 02:44:49,921 - INFO - Cosine-Similarity:	Pearson: 0.0098	Spearman: -0.0195
2025-11-13 02:44:49,925 - INFO - Save model to /Users/don/DocumentsMac/Codes/itsm-insight-nexus/backend-python/scripts/finetuning/models/all-mpnet-finetuned
2025-11-13 02:44:49,921 - INFO - Cosine-Similarity:	Pearson: 0.0098	Spearman: -0.0195
2025-11-13 02:44:49,925 - INFO - Save model to /Users/don/DocumentsMac/Codes/itsm-insight-nexus/backend-python/scripts/finetuning/models/all-mpnet-finetuned
2025-11-13 02:44:57,831 - INFO - EmbeddingSimilarityEvaluator: Evaluating the model on the itsm-eval dataset in epoch 1.0 after 10 steps:
2025-11-13 02:44:57,831 - INFO - EmbeddingSimilarityEvaluator: Evaluating the model on the itsm-eval dataset in epoch 1.0 after 10 steps:
2025-11-13 02:44:58,344 - INFO - Cosine-Similarity:	Pearson: 0.0166	Spearman: -0.0130
2025-11-13 02:4

## 8. Save Training Metadata

In [9]:
# Save metadata
metadata = {
    "base_model": CONFIG['base_model'],
    "training_date": datetime.now().isoformat(),
    "epochs": CONFIG['epochs'],
    "batch_size": CONFIG['batch_size'],
    "learning_rate": CONFIG['learning_rate'],
    "num_train_examples": len(train_examples),
    "num_eval_examples": len(eval_examples),
    "num_positive_pairs": len(positive_pairs),
    "num_negative_pairs": len(negative_pairs),
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

metadata_path = os.path.join(output_path, 'training_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"💾 Model saved to: {output_path}")
print(f"📝 Metadata saved to: {metadata_path}")

## 9. Quick Evaluation

In [10]:
# Load the fine-tuned model
finetuned_model = SentenceTransformer(output_path)

# Test with example tickets
if positive_pairs:
    test_pair = positive_pairs[0]
    
    # Generate embeddings
    emb1 = finetuned_model.encode(test_pair['text1'])
    emb2 = finetuned_model.encode(test_pair['text2'])
    
    # Calculate similarity
    from sklearn.metrics.pairwise import cosine_similarity
    similarity = cosine_similarity([emb1], [emb2])[0][0]
    
    print("\n📊 Quick Test:")
    print(f"Category: {test_pair['category1']}")
    print(f"Ticket 1: {test_pair['ticket1_id']}")
    print(f"Ticket 2: {test_pair['ticket2_id']}")
    print(f"\nSimilarity Score: {similarity:.4f}")
    print(f"Expected: High (same category)")
    
    if similarity > 0.7:
        print("✅ Good! Model correctly identifies similar tickets")
    elif similarity > 0.5:
        print("⚠️  Moderate similarity - model needs more training")
    else:
        print("❌ Low similarity - model may need different approach")

## 10. Next Steps

Now that you have fine-tuned the all-mpnet-base-v2 model, you can:

1. **Use the model locally**:
   ```python
   from sentence_transformers import SentenceTransformer
   model = SentenceTransformer('scripts/finetuning/models/all-mpnet-finetuned')
   embeddings = model.encode(["ticket text here"])
   ```

2. **Update your embedding service** (`app/services/embedding_service.py`) to use this fine-tuned model instead of LM Studio

3. **Run full evaluation** to compare fine-tuned model with LM Studio models:
   ```bash
   python scripts/performance_eval/compare_models.py
   ```

4. **Regenerate embeddings** for all tickets using the fine-tuned model:
   ```bash
   python scripts/populate_embeddings.py
   ```

## 11. Load and Test Fine-tuned Model

In [ ]:
# You can reload the model anytime with:
print("Loading fine-tuned model...")
finetuned = SentenceTransformer(output_path)
print(f"✅ Fine-tuned model loaded from: {output_path}")
print(f"Embedding dimension: {finetuned.get_sentence_embedding_dimension()}")